In [ ]:
import os
import sys
import json
import numpy as np

sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")

from utls.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability, estimate_split_half_reliability
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utls.reliability import compute_power_curve, compute_power_curve
from utls.plotting import plot_power_curve#, plot_groupwise_item_response_scatter


import os, random
import numpy as np
import pandas as pd

def compute_itemwise_split_half_reliability_by_isi(
    exps,
    criterion=1,
    n_splits=100,
    random_seed=42,
    isi_values=None,
):
    """
    Compute split-half reliability for itemwise responses, separately for each ISI.

    Args:
        exps (list[pd.DataFrame]): one dataframe per participant
        criterion (int): threshold for counting a 'yes' response
        n_splits (int): number of random splits
        random_seed (int): random seed for reproducibility
        isi_values (list or None): ISIs to include (if None, inferred from data)

    Returns:
        dict[isi] -> reliability results for hits and FAs
    """
    random.seed(random_seed)
    np.random.seed(random_seed)

    # infer ISIs
    if isi_values is None:
        all_isis = sorted({isi for df in exps for isi in df['isi'].unique() if pd.notna(isi)})
    else:
        all_isis = isi_values

    results = {}

    for isi in all_isis:
        all_signal_items = set()
        all_noise_items = set()
        participant_data_signal = []
        participant_data_noise = []

        for df in exps:
            yt_ids    = [os.path.basename(x) for x in df['stimulus']]
            responses = df['response'].tolist()
            repeats   = df['repeat'].tolist()
            isis      = df['isi'].tolist()

            signal_row = {}
            noise_row = {}

            for yt, resp, repeat, curr_isi in zip(yt_ids, responses, repeats, isis):
                if pd.isna(resp) or pd.isna(yt):
                    continue
                if curr_isi != isi:
                    continue

                is_yes = int(int(resp) > criterion)

                if repeat == 'true':
                    signal_row[yt] = is_yes
                    all_signal_items.add(yt)
                elif repeat == 'false':
                    noise_row[yt] = is_yes
                    all_noise_items.add(yt)

            participant_data_signal.append(signal_row)
            participant_data_noise.append(noise_row)

        signal_df = pd.DataFrame(participant_data_signal, columns=sorted(all_signal_items)).astype("float")
        noise_df  = pd.DataFrame(participant_data_noise,  columns=sorted(all_noise_items)).astype("float")

        signal_r, signal_std = estimate_split_half_reliability(signal_df, n_splits=n_splits, seed=random_seed)
        noise_r,  noise_std  = estimate_split_half_reliability(noise_df,  n_splits=n_splits, seed=random_seed)

        results[isi] = {
            'split_half_reliability': {
                'hits': (signal_r, signal_std),
                'false_alarms': (noise_r, noise_std)
            },
            'itemwise_responses': {
                'hits': signal_df,
                'false_alarms': noise_df
            }
        }

    return results

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_dprime_by_isi(
    df_per_subject,
    stimulus_set=None,
    show_mean_sem=True,
    log_scale=False,
    save_path=None
):
    """
    Plot d' versus ISI across participants.

    Args:
        df_per_subject (pd.DataFrame): Must have columns ['subject', 'isi', 'd_prime'].
        stimulus_set (str or None): Title label for the plot.
        show_mean_sem (bool): Whether to overlay mean ± SEM curve.
        log_scale (bool): If True, use log10 scale for the y-axis.
        save_path (str or None): If given, save figure to this path instead of showing it.
    """

    plt.figure(figsize=(10, 6))

    # Individual subject curves
    for subject_id, subj_df in df_per_subject.groupby("subject"):
        plt.plot(subj_df["isi"], subj_df["d_prime"], marker='o', alpha=0.4)

    # Mean ± SEM overlay
    if show_mean_sem:
        grouped = df_per_subject.groupby("isi")["d_prime"]
        mean_d = grouped.mean()
        sem_d  = grouped.sem()

        plt.errorbar(
            mean_d.index, mean_d.values, yerr=sem_d.values,
            fmt='-o', color='black', capsize=3, linewidth=2,
            label="mean ± SEM"
        )
        plt.legend()

    plt.xlabel("ISI (Interstimulus Interval)")
    plt.ylabel("d′ (sensitivity index)")
    title = f"{stimulus_set}: d′ vs ISI Across Participants" if stimulus_set else "d′ vs ISI Across Participants"
    plt.title(title)
    plt.grid(True, which='both', ls='--', alpha=0.5)
    plt.tight_layout()

    # Scaling
    if log_scale:
        plt.yscale("log")
        plt.ylabel("log₁₀(d′)")
        # Avoid negative or zero values for log
        min_nonzero = df_per_subject["d_prime"][df_per_subject["d_prime"] > 0].min()
        if np.isfinite(min_nonzero):
            plt.ylim(bottom=min_nonzero * 0.8)
    else:
        plt.ylim([0, 6])

    # Save or show
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close()
    else:
        plt.show()

from scipy.stats import norm
import numpy as np
import pandas as pd

def compute_dprime_from_pairs(df, selected_pairs, criterion=1, epsilon=1e-4):
    """
    Compute d′ using (nonrepeat, repeat) index pairs.

    This version automatically handles:
      - DataFrames with or without 'orig_index'
      - Mixed response datatypes (str, int, float)
      - Missing or invalid indices gracefully

    Args:
        df : pd.DataFrame
            Must contain a 'response' column.
        selected_pairs : list of tuple(int, int)
            Each tuple = (nonrepeat_idx, repeat_idx) from the *original* trial order.
        criterion : int or float
            Threshold above which a response counts as "yes".
        epsilon : float
            Small value for clipping extreme hit/FA rates to avoid infinite z-scores.

    Returns:
        tuple (dprime, hit_rate, fa_rate)
    """

    if "response" not in df.columns:
        raise ValueError("DataFrame must contain a 'response' column.")

    # Determine usable index (prefer 'orig_index' if present)
    if "orig_index" in df.columns:
        df_lookup = df.set_index("orig_index", drop=False)
        use_iloc = False
    else:
        df_lookup = df
        use_iloc = True

    # Collect indices for repeats and non-repeats
    repeat_idxs = [pair[1] for pair in selected_pairs]
    nonrepeat_idxs = [pair[0] for pair in selected_pairs]

    # Retrieve responses safely
    try:
        if use_iloc:
            hit_responses = df_lookup.iloc[repeat_idxs]["response"].astype(float).dropna()
            fa_responses  = df_lookup.iloc[nonrepeat_idxs]["response"].astype(float).dropna()
        else:
            hit_responses = df_lookup.loc[repeat_idxs, "response"].astype(float).dropna()
            fa_responses  = df_lookup.loc[nonrepeat_idxs, "response"].astype(float).dropna()
    except Exception as e:
        # Fallback in case of indexing mismatch
        hit_responses = df["response"].iloc[[p[1] for p in selected_pairs if p[1] < len(df)]].astype(float).dropna()
        fa_responses  = df["response"].iloc[[p[0] for p in selected_pairs if p[0] < len(df)]].astype(float).dropna()

    # Compute hit and false alarm rates
    hits = (hit_responses > criterion).sum()
    fas  = (fa_responses > criterion).sum()

    n_hits = len(hit_responses)
    n_fas  = len(fa_responses)

    # Rates with epsilon correction
    hit_rate = np.clip(hits / n_hits, epsilon, 1 - epsilon) if n_hits > 0 else np.nan
    fa_rate  = np.clip(fas  / n_fas,  epsilon, 1 - epsilon) if n_fas  > 0 else np.nan

    # Compute d'
    if np.isfinite(hit_rate) and np.isfinite(fa_rate):
        dprime = norm.ppf(hit_rate) - norm.ppf(fa_rate)
    else:
        dprime = np.nan

    return dprime, hit_rate, fa_rate


def load_results_with_exclusion_no_dropping(
    results_dir,
    min_dprime=1.0,
    min_trials=120,
    skip_len60=True,
    verbose=False,
    return_skipped=False,
):
    """
    Load experiment results and filter participants based on ISI=0 d′,
    without modifying or dropping any trials.

    Args:
        results_dir (str): directory containing participant result files
        min_dprime (float): minimum d′ at ISI=0 required to keep participant
        min_trials (int): minimum total number of trials required
        skip_len60 (bool): skip 60-trial runs
        verbose (bool): print detailed progress
        return_skipped (bool): if True, also return skipped participants

    Returns:
        filtered_exps, filtered_seqs, filtered_fnames
        (optionally skipped_exps, skipped_seqs, skipped_fnames if return_skipped=True)
    """
    exps, seqs, fnames = load_results(
        results_dir=results_dir,
        min_trials=min_trials,
        skip_len60=skip_len60
    )

    np.random.seed(42)  # for reproducibility

    filtered_exps, filtered_seqs, filtered_fnames = [], [], []
    skipped_exps, skipped_seqs, skipped_fnames = [], [], []

    for df, seq_file, fname in zip(exps, seqs, fnames):
        df = df.copy()

        if "isi" not in df.columns or "yt_id" not in df.columns:
            if verbose:
                print(f"[{fname}] Skipped: missing required columns")
            continue

        # Compute ISI=0 d′ for screening
        isi0_repeat_rows = df[df["isi"] == 0.0]
        paired_indices = []

        yt_map = {}
        for i, yt in enumerate(df["yt_id"]):
            yt_map.setdefault(yt, []).append(i)

        for i in isi0_repeat_rows.index:
            yt = df.loc[i, "yt_id"]
            prior_indices = [j for j in yt_map[yt] if j < i]
            if not prior_indices:
                continue
            paired_indices.append((i, prior_indices[-1]))  # (non-repeat, repeat)

        if len(paired_indices) < 2:
            if verbose:
                print(f"[{fname}] Skipped: fewer than 2 ISI=0 repeat pairs")
            skipped_exps.append(df)
            skipped_seqs.append(seq_file)
            skipped_fnames.append(fname)
            continue

        # Compute d' for all ISI=0 pairs (no dropping)
        dprime, hit_rate, fa_rate = compute_dprime_from_pairs(df, paired_indices)
        print(f"[{fname}] ISI=0 d' = {dprime:.2f} (HR={hit_rate:.2f}, FAR={fa_rate:.2f})")

        if dprime >= min_dprime:
            filtered_exps.append(df)
            filtered_seqs.append(seq_file)
            filtered_fnames.append(fname)
        else:
            if verbose:
                print(f"[{fname}] Excluded: d′ < {min_dprime}")
                print(f"[{fname}] Excluded: n={len(df)}")

            skipped_exps.append(df)
            skipped_seqs.append(seq_file)
            skipped_fnames.append(fname)

    if return_skipped:
        return (
            filtered_exps, filtered_seqs, filtered_fnames,
            skipped_exps, skipped_seqs, skipped_fnames
        )

    return filtered_exps, filtered_seqs, filtered_fnames


def load_results_with_exclusion_no_dropping_diagnostic(
    results_dir,
    min_dprime=1.0,
    min_trials=120,
    skip_len60=True,
    verbose=True,
    return_skipped=False,
):
    """
    Load experiment results and filter participants based on ISI=0 d′,
    without modifying or dropping any trials.
    Includes detailed per-participant diagnostics (counts, pairs, hits/FAs).

    Args:
        results_dir (str)
        min_dprime (float)
        min_trials (int)
        skip_len60 (bool)
        verbose (bool)
        return_skipped (bool)

    Returns:
        filtered_exps, filtered_seqs, filtered_fnames
        (optionally skipped_exps, skipped_seqs, skipped_fnames)
    """

    # Load base experiment data
    exps, seqs, fnames = load_results(
        results_dir=results_dir,
        min_trials=min_trials,
        skip_len60=skip_len60
    )

    np.random.seed(42)

    filtered_exps, filtered_seqs, filtered_fnames = [], [], []
    skipped_exps, skipped_seqs, skipped_fnames = [], [], []

    print("\n=== Participant Diagnostics ===\n")

    for df, seq_file, fname in zip(exps, seqs, fnames):
        df = df.copy().reset_index(drop=True)  # ensure clean positional index

        if "isi" not in df.columns or "yt_id" not in df.columns:
            if verbose:
                print(f"[{fname}] ⚠️ Skipped: missing required columns")
            continue

        # Total trial counts
        n_total = len(df)
        n_isi0  = (np.isclose(df["isi"].astype(float), 0.0, atol=1e-3)).sum()

        # Select ISI=0 repeat trials robustly
        repeat_mask = (np.isclose(df["isi"].astype(float), 0.0, atol=1e-3)) & (
            df["repeat"].astype(str).str.lower().isin(["true", "1", "yes"])
        )
        isi0_repeat_rows = df[repeat_mask]
        paired_indices = []

        # Map yt_id → list of positional indices
        yt_map = {}
        for i, yt in enumerate(df["yt_id"]):
            yt_map.setdefault(yt, []).append(i)

        # Build (repeat, nonrepeat) pairs
        for i in isi0_repeat_rows.index:
            yt = df.iloc[i]["yt_id"]
            prior_indices = [j for j in yt_map[yt] if j < i]
            if not prior_indices:
                continue
            paired_indices.append((i, prior_indices[-1]))

        n_pairs = len(paired_indices)

        # Handle cases with too few pairs
        if n_pairs < 2:
            if verbose:
                print(
                    f"[{fname}] ⚠️ Skipped: fewer than 2 ISI=0 pairs "
                    f"(n_pairs={n_pairs}, n_isi0={n_isi0}, total={n_total})"
                )
            skipped_exps.append(df)
            skipped_seqs.append(seq_file)
            skipped_fnames.append(fname)
            continue

        # Compute d′, HR, FAR
        dprime, hit_rate, fa_rate = compute_dprime_from_pairs(df, paired_indices)

        # Diagnostic counts
        repeat_idxs = [p[0] for p in paired_indices]
        nonrepeat_idxs = [p[1] for p in paired_indices]

        n_hits = len(df.iloc[repeat_idxs]["response"].dropna())
        n_fas  = len(df.iloc[nonrepeat_idxs]["response"].dropna())

        print(
            f"[{fname}] n_total={n_total}, n_isi0={n_isi0}, "
            f"n_pairs={n_pairs}, n_hits={n_hits}, n_fas={n_fas}"
        )
        print(f"    ISI=0 d′={dprime:.2f}, HR={hit_rate:.2f}, FAR={fa_rate:.2f}")

        # Apply inclusion rule
        if dprime >= min_dprime and np.isfinite(dprime):
            filtered_exps.append(df)
            filtered_seqs.append(seq_file)
            filtered_fnames.append(fname)
        else:
            if verbose:
                print(f"    → Excluded: d′ < {min_dprime} or NaN")
            skipped_exps.append(df)
            skipped_seqs.append(seq_file)
            skipped_fnames.append(fname)

        print("-" * 60)

    print("\n=== Summary ===")
    print(f"Included: {len(filtered_exps)} participants")
    print(f"Excluded: {len(skipped_exps)} participants\n")

    if return_skipped:
        return (
            filtered_exps, filtered_seqs, filtered_fnames,
            skipped_exps, skipped_seqs, skipped_fnames
        )

    return filtered_exps, filtered_seqs, filtered_fnames

import json

def split_by_musicianship(exps):
    """
    Split participant DataFrames into experienced (>6 years) and less experienced musicians.

    Rules:
    - If a DataFrame only has 'musician_info' (no 'is_musician'), classify as beginner (<5 years).
    - Otherwise, parse 'is_musician' JSON and classify by the 'years_playing' field.
    """

    experienced = []   # >6 years
    beginner = []      # ≤6 years or missing

    for exp in exps:
        # Case 1: legacy participant with only 'musician_info'
        if "is_musician" not in exp.columns:
            beginner.append(exp)
            continue

        # Parse the JSON string (some jsPsych exports are stored as stringified JSON)
        try:
            info = json.loads(exp["is_musician"].iloc[0])
        except Exception as e:
            print("Warning: could not parse is_musician:", e)
            beginner.append(exp)
            continue

        # Safely extract years_playing
        years_playing = info.get("years_playing", "").strip().lower()

        if years_playing in ["7–10 years", "more than 10 years"]:
            experienced.append(exp)
        else:
            beginner.append(exp)

    return experienced, beginner

In [ ]:
# results = set(glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/ind-nature-len120/*csv"))
# results = list(results)

tasks = ["env-sounds" ,"global-music-len120", "atexts-len120", "nhs-region-len120"]
which_task = tasks[0] # "global-music-len120", "atexts-len120" "nhs-region-len120"
base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/len120_multi/"
seqs_paths = {"env-sounds": "mem_exp_ind-nature_2025", 
              "global-music-len120": "global-music-2025-n_80",
              "atexts-len120": "mem_exp_atexts_2025",
              "nhs-region-len120": "nhs-region-n_80"}


hr_task_name = {"env-sounds": "Industrial and Nature", 
              "global-music-len120": "Globalized Music",
              "atexts-len120": "Auditory Textures",
              "nhs-region-len120": " 'Natural History of Song' "}

exps, seqs, fnames = load_results_with_exclusion_no_dropping(f"/mindhive/mcdermott/www/bjmedina/experiments/{which_task}/results/{which_task}/len120_multi",
                                                    min_dprime=2,
                                                    min_trials=120,
                                                    skip_len60=True,
                                                    verbose=True,
                                                    return_skipped=False)



experienced, beginner = split_by_musicianship(exps)
print(base_path.format(seqs_paths[which_task]))
move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

print("Number of participants used in analysis:", len(exps))


safe_name = which_task.lower().replace(" ", "_")  # e.g., "globalized_music"
save_dir = os.path.join("/om2/user/bjmedina/auditory-memory/memory/figures/human-results/multi-isi", safe_name)

ensure_dir(save_dir)
print(save_dir)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

def simulate_chance_dprime(exps, yes_rate=0.5, n_runs=1000, seed=42):
    """
    Simulate random yes/no responses on the actual experiment sequences
    and compute the overall chance d' distribution.

    Args:
        exps (list[pd.DataFrame]): list of experiment dataframes
        yes_rate (float): probability of responding "yes"
        n_runs (int): number of simulation runs
        seed (int): RNG seed

    Returns:
        np.ndarray of simulated d' values across runs
    """
    rng = np.random.default_rng(seed)
    dprimes = []

    for df in exps:
        df = df.copy()
        if "repeat" not in df.columns:
            continue

        # Identify which trials are repeats (signals)
        repeat_mask = df["repeat"].astype(str).str.lower().isin(["true", "1", "yes"])
        #print(repeat_mask)
        n_signal = repeat_mask.sum()
        n_noise = (~repeat_mask).sum()
        #print(n_signal, n_noise)
        if n_signal == 0 or n_noise == 0:
            continue

        n_trials = len(df)

        for _ in range(n_runs):
            n_yes = int(np.floor(len(df) * yes_rate))
            responses = np.zeros(len(df), dtype=int)
            responses[:n_yes] = 1
            rng.shuffle(responses)
            #print(responses, np.sum(responses))
            #print(responses)
            # Compute hit/FA
            hits = responses[repeat_mask].sum()
            fas  = responses[~repeat_mask].sum()

            hit_rate = hits / n_signal
            fa_rate  = fas  / n_noise

            # clip to avoid infinities
            eps = 1e-5
            hit_rate = np.clip(hit_rate, eps, 1 - eps)
            fa_rate  = np.clip(fa_rate,  eps, 1 - eps)

            dprime = norm.ppf(hit_rate) - norm.ppf(fa_rate)
            dprimes.append(dprime)

            #input()

    return np.array(dprimes)

In [ ]:
chance_summary = simulate_chance_dprime(exps, yes_rate=1/3, n_runs=1000)
print(np.mean(chance_summary), np.std(chance_summary))

In [ ]:
chance_summary = simulate_chance_dprime(exps, yes_rate=1/2, n_runs=1000)
print(np.mean(chance_summary), np.std(chance_summary))

In [ ]:
chance_summary = simulate_chance_dprime(exps, yes_rate=2/3, n_runs=1000)
print(np.mean(chance_summary), np.std(chance_summary))

In [ ]:
chance_summary = simulate_chance_dprime(exps, yes_rate=0.9999, n_runs=1000)
print(np.mean(chance_summary), np.std(chance_summary))

In [ ]:
# ====================================================
# Sweep over different "yes rates"
# ====================================================

yes_rates = np.linspace(0.05, 0.95, 10)  # from 5% to 95%
mean_dprimes, std_dprimes = [], []

for r in yes_rates:
    dp = simulate_chance_dprime(exps, yes_rate=r, n_runs=500)
    mean_dprimes.append(np.mean(dp))
    std_dprimes.append(np.std(dp))

# ====================================================
# Plot d' vs yes_rate
# ====================================================
plt.figure(figsize=(7,5))
plt.plot(yes_rates, mean_dprimes, '-o', lw=2)
plt.fill_between(yes_rates,
                 np.array(mean_dprimes)-np.array(std_dprimes),
                 np.array(mean_dprimes)+np.array(std_dprimes),
                 alpha=0.2)

plt.axhline(0, color='k', linestyle='--')
plt.xlabel("Proportion of 'Yes' Responses (Bias)")
plt.ylabel("Simulated Chance d′")
plt.title("Chance d′ as a Function of Response Bias")
plt.grid(True)
plt.show()